In [ ]:
from model_fitting import fit_compare_to_threshold_model
from nwb_utils import NWBUtils

nwb_file_name='/root/capsule/scratch/general_behavior/698694/698694_2024-04-17_12-21-55.nwb'
nwb_data=NWBUtils.read_behavior_nwb(nwb_full_path=nwb_file_name)


In [ ]:
fitted_data=fit_compare_to_threshold_model(nwb_data,
                                        reset_on_switch=True,
                                        include_bias=True,
                                        save_results=True,
                                        save_folder=save_folder
                                        )

In [1]:
!OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 \
 python /root/capsule/src/aind_dft_ephys_analysis/run_ctt_grid_mp.py


Found 11076 NWB files under: /root/capsule/scratch/general_behavior
Saving JSON results under: /root/capsule/scratch/CTT_grid_json_resetmode_threshold
Grid size per session: 48
Total models: 531648
Running with 12 processes (spawn)
FILES_PER_TASK=3
Submitting 3692 tasks
Found behavior NWB: /root/capsule/scratch/general_behavior/698694/698694_2024-04-17_12-21-55.nwb
Found behavior NWB: /root/capsule/scratch/general_behavior/698694/698694_2024-04-22_10-56-51.nwb
Found behavior NWB: /root/capsule/scratch/general_behavior/703548/703548_2024-03-11_12-01-57.nwb
Found behavior NWB: /root/capsule/scratch/general_behavior/703548/703548_2024-03-13_11-30-07.nwb
Successfully read behavior NWB from: /root/capsule/scratch/general_behavior/703548/703548_2024-03-11_12-01-57.nwb
No valid trials after removing 'no response'.
No valid trials after removing 'no response'.
No valid trials after removing 'no response'.
Successfully read behavior NWB from: /root/capsule/scratch/general_behavior/698694/698694

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import pandas as pd


def load_ctt_results_jsons_to_dataframe(
    results_root: Union[str, Path],
    *,
    recursive: bool = True,
    keep_latents: bool = False,
) -> pd.DataFrame:
    """
    Load all compare-to-threshold JSON result files under a results folder and
    concatenate them into a single pandas DataFrame.

    This is designed for the folder structure produced by your runner:
        CTT_grid_json_only_by_animal/
            <animal_id>/
                reset_on_switch_{True|False}/
                    include_bias_{True|False}/
                        <session_id>__<model_name>_fit_results.json

    Parameters
    ----------
    results_root
        Path to the root folder (e.g. /root/capsule/scratch/CTT_grid_json_only_by_animal).
    recursive
        If True, searches recursively for all *.json files under results_root.
        If False, only searches one level below results_root (not recommended here).
    keep_latents
        If True, keep the fitted_latent_variables field (which can be large) as
        a column containing dicts.
        If False, drop fitted_latent_variables to keep the DataFrame light-weight.

    Returns
    -------
    pd.DataFrame
        One row per JSON file. Nested dicts are flattened into columns using
        dotted keys, e.g.:
            fitted_params.alpha
            metadata.reset_on_switch

        If keep_latents=True, fitted_latent_variables is kept as a dict column
        (not flattened by default, since it can be huge).
    """

    root = Path(results_root)

    if not root.exists():
        raise FileNotFoundError(f"results_root does not exist: {root}")

    if recursive:
        json_files = sorted(root.rglob("*.json"))
    else:
        json_files = sorted(root.glob("*.json"))

    records: List[Dict[str, Any]] = []

    for jf in json_files:
        try:
            with open(jf, "r") as f:
                data = json.load(f)

            # Attach path-derived metadata that is useful for grouping
            # Expected layout: root / animal_id / reset_on_switch_X / include_bias_Y / file.json
            rel_parts = jf.relative_to(root).parts
            animal_id = rel_parts[0] if len(rel_parts) >= 1 else None

            reset_on_switch = None
            include_bias = None
            if len(rel_parts) >= 3:
                # e.g. "reset_on_switch_True"
                if rel_parts[1].startswith("reset_on_switch_"):
                    reset_on_switch = rel_parts[1].split("reset_on_switch_", 1)[1]
                    if reset_on_switch in ("True", "False"):
                        reset_on_switch = (reset_on_switch == "True")

                # e.g. "include_bias_False"
                if rel_parts[2].startswith("include_bias_"):
                    include_bias = rel_parts[2].split("include_bias_", 1)[1]
                    if include_bias in ("True", "False"):
                        include_bias = (include_bias == "True")

            data["_json_path"] = str(jf)
            data["_animal_id_from_path"] = animal_id
            data["_reset_on_switch_from_path"] = reset_on_switch
            data["_include_bias_from_path"] = include_bias

            # Optionally drop latents to avoid huge DF
            if not keep_latents and "fitted_latent_variables" in data:
                data.pop("fitted_latent_variables", None)

            records.append(data)

        except Exception as e:
            # Keep a minimal record for debugging failed loads
            records.append(
                {
                    "_json_path": str(jf),
                    "_load_error": repr(e),
                }
            )

    # Flatten nested dicts (except fitted_latent_variables if kept)
    df = pd.json_normalize(records, sep=".")

    # Optional: make common columns more convenient (if they exist)
    # If the JSON already contains these, the path-derived ones can be used as backup.
    if "animal_id" not in df.columns and "_animal_id_from_path" in df.columns:
        df["animal_id"] = df["_animal_id_from_path"]

    # If you want a single authoritative reset/include columns, prefer JSON metadata if present
    if "metadata.reset_on_switch" in df.columns and "_reset_on_switch_from_path" in df.columns:
        df["reset_on_switch"] = df["metadata.reset_on_switch"].where(
            df["metadata.reset_on_switch"].notna(), df["_reset_on_switch_from_path"]
        )
    elif "_reset_on_switch_from_path" in df.columns:
        df["reset_on_switch"] = df["_reset_on_switch_from_path"]

    if "metadata.include_bias" in df.columns and "_include_bias_from_path" in df.columns:
        df["include_bias"] = df["metadata.include_bias"].where(
            df["metadata.include_bias"].notna(), df["_include_bias_from_path"]
        )
    elif "_include_bias_from_path" in df.columns:
        df["include_bias"] = df["_include_bias_from_path"]

    return df


# Example usage:
# df = load_ctt_results_jsons_to_dataframe(
#     "/root/capsule/scratch/CTT_grid_json_only_by_animal",
#     keep_latents=False,
# )
# df.head()


In [ ]:

df = load_ctt_results_jsons_to_dataframe(
    "/root/capsule/scratch/CTT_grid_json_only_by_animal",
    keep_latents=False,
)
df.head()

In [ ]:
from __future__ import annotations

from typing import Sequence, Optional, Dict, Any, List, Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_ctt_metric_distributions_by_condition(
    df: pd.DataFrame,
    *,
    metrics: Optional[Sequence[str]] = None,
    condition_cols: Sequence[str] = ("reset_on_switch", "include_bias"),
    auto_train_stages: Optional[Union[Sequence[Any], Any]] = None,
    dropna: bool = True,
    bins: int = 50,
    max_cols: int = 3,
) -> Dict[str, Any]:
    """
    Plot distribution (histogram) of key fit metrics/parameters for each
    (reset_on_switch, include_bias) condition, optionally filtering by
    auto_train_stage.

    Parameters
    ----------
    df
        DataFrame produced by your JSON loader.
    metrics
        Columns to plot.
    condition_cols
        Two columns defining the condition grid.
    auto_train_stages
        None → use all sessions
        single value or list → only include those stages
    dropna
        Drop NaNs per metric before plotting.
    bins
        Histogram bins.
    max_cols
        Maximum subplot columns.

    Returns
    -------
    dict mapping metric -> figure info
    """

    if metrics is None:
        metrics = [
            "log_likelihood",
            "aic",
            "bic",
            "fitted_params.alpha",
            "fitted_params.threshold",
            "fitted_params.beta",
            "fitted_params.bias",
        ]

    # -----------------------------
    # Filter by auto_train_stage
    # -----------------------------
    df_plot = df.copy()

    if auto_train_stages is not None:
        if "auto_train_stage" not in df_plot.columns:
            raise KeyError("Column 'auto_train_stage' not found in DataFrame.")

        if not isinstance(auto_train_stages, (list, tuple, set)):
            auto_train_stages = [auto_train_stages]

        df_plot = df_plot[df_plot["auto_train_stage"].isin(auto_train_stages)]

        print(
            f"Filtering auto_train_stage in {list(auto_train_stages)} "
            f"-> {len(df_plot)} rows remaining"
        )

    # -----------------------------
    # Validate condition columns
    # -----------------------------
    for c in condition_cols:
        if c not in df_plot.columns:
            raise KeyError(f"Missing condition column '{c}'")

    # Ensure consistent ordering of groups
    def _sort_key(v):
        if isinstance(v, bool):
            return int(v)
        if pd.isna(v):
            return 2
        return str(v)

    group_keys = sorted(
        df_plot[list(condition_cols)].drop_duplicates().itertuples(index=False, name=None),
        key=lambda t: tuple(_sort_key(x) for x in t),
    )

    out: Dict[str, Any] = {}

    stage_str = (
        "All stages"
        if auto_train_stages is None
        else f"Stages={list(auto_train_stages)}"
    )

    for metric in metrics:
        if metric not in df_plot.columns:
            continue

        n_groups = len(group_keys)
        ncols = min(max_cols, n_groups) if n_groups > 0 else 1
        nrows = int(np.ceil(n_groups / ncols)) if n_groups > 0 else 1

        fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                                 figsize=(5.0 * ncols, 3.8 * nrows))
        axes = np.array(axes).reshape(-1)

        for i, key in enumerate(group_keys):
            ax = axes[i]

            mask = np.ones(len(df_plot), dtype=bool)
            for col, val in zip(condition_cols, key):
                mask &= (df_plot[col] == val)

            s = df_plot.loc[mask, metric]
            if dropna:
                s = s.dropna()

            if len(s) == 0:
                ax.text(0.5, 0.5, "No data", ha="center", va="center")
                ax.set_axis_off()
                continue

            ax.hist(s.values, bins=bins)
            ax.set_title(
                f"{metric}\n"
                f"{condition_cols[0]}={key[0]}, "
                f"{condition_cols[1]}={key[1]}\n"
                f"{stage_str}"
            )
            ax.set_xlabel(metric)
            ax.set_ylabel("Count")
            ax.grid(True, alpha=0.25)

            ax.text(
                0.98,
                0.98,
                f"n={len(s)}\nmean={np.mean(s):.3g}\nmed={np.median(s):.3g}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=10,
                family="monospace",
            )

        for j in range(i + 1, len(axes)):
            axes[j].set_axis_off()

        fig.tight_layout()
        out[metric] = {"fig": fig, "axes": axes, "groups": group_keys}

    return out


In [ ]:
plot_ctt_metric_distributions_by_condition(df, auto_train_stages=['GRADUATED'])